In [23]:
import xmlrpc.client
import pickle
from PIL import Image
from pylab import *
import numpy as np
import datetime
import glob
import time
# Режим отображения графиков внутри ноутбука
%matplotlib inline

In [3]:
import xmlrpc.client
stats2 = xmlrpc.client.ServerProxy("http://localhost:8018", allow_none=True)

# 1. Кто это?
print("type:", type(stats2))

# 2. Что за метод get_log?
print("get_log:", stats2.get_log)

# 3. Прямой вызов без аргументов:
print("no args:", stats2.get_log())

# 4. Прямой вызов с event_type:
print("with kwarg:", stats2.get_log(event_type='too_many_requests'))

type: <class 'xmlrpc.client.ServerProxy'>
get_log: <xmlrpc.client._Method object at 0x741ebbe51f30>
no args: [['ping', '2026-10-19 18:06:31', None], ['ping', '2026-10-19 18:06:31', 0.04341745376586914], ['now', '2026-10-19 18:06:31', None], ['now', '2026-10-19 18:06:31', 0.017699003219604492], ['type', '2026-10-19 18:06:31', None], ['type', '2026-10-19 18:06:31', 0.02519679069519043], ['type', '2026-10-19 18:06:31', None], ['type', '2026-10-19 18:06:31', 0.016265869140625], ['type', '2026-10-19 18:06:31', None], ['type', '2026-10-19 18:06:31', 0.13184571266174316], ['ping', '2026-10-19 18:07:22', None], ['ping', '2026-10-19 18:07:22', 0.04461979866027832], ['now', '2026-10-19 18:07:22', None], ['now', '2026-10-19 18:07:22', 0.015058040618896484], ['type', '2026-10-19 18:07:22', None], ['type', '2026-10-19 18:07:22', 0.018667936325073242], ['type', '2026-10-19 18:07:22', None], ['type', '2026-10-19 18:07:22', 0.02261519432067871], ['type', '2026-10-19 18:07:22', None], ['type', '2026-10

TypeError: _Method.__call__() got an unexpected keyword argument 'event_type'

In [2]:
import importlib
import xmlrpc.client
importlib.reload(xmlrpc.client)

stats2 = xmlrpc.client.ServerProxy("http://localhost:8018", allow_none=True)
print(stats2.system.listMethods())
print(stats2.get_log(event_type='too_many_requests'))

['add_log', 'get_log', 'ping_stats', 'system.listMethods', 'system.methodHelp', 'system.methodSignature']


TypeError: _Method.__call__() got an unexpected keyword argument 'event_type'

In [ ]:
print("Файлы в папке:", glob.glob('log*.csv'))

In [24]:
import xmlrpc.client
s = xmlrpc.client.ServerProxy("http://localhost:8018")
print(s.system.listMethods())

['add_log', 'get_log', 'ping_stats', 'system.listMethods', 'system.methodHelp', 'system.methodSignature']


In [25]:
# Подключение к серверу
#server = xmlrpc.client.ServerProxy("http://localhost:8008")
server = xmlrpc.client.ServerProxy("http://localhost:8009")
#stats = xmlrpc.client.ServerProxy("http://localhost:8018", allow_none=True)
stats2 = xmlrpc.client.ServerProxy("http://localhost:8018", allow_none=True)

In [26]:
# Тесты
print(server.system.listMethods())
print('Ping:', server.ping())
print('Server datetime:', server.now())
print('View, type, value:', server.type(2))
print('View, type, value:', server.type(2.))
print('View, type, value:', server.type('My string'))
print('View, type, value:', server.type("My string"))
print('View, type, value:', server.type([1,2,3]))
print('View, type, value:', server.type(["one", "two", "three"]))
print('View, type, value:', server.type((1,2,"3")))
print('Sum 2 + 3 :', server.sum(2, 3))
print('Pow 2^3: ', server.pow(2, 3))

['binarize_image', 'black_list_check', 'cloud_percent', 'color_inversion', 'flip_vertical', 'now', 'ping', 'pow', 'proxy_ping', 'sum', 'system.listMethods', 'system.methodHelp', 'system.methodSignature', 'type']
Ping: True
Server datetime: 20261019T18:40:44
View, type, value: ['2', "<class 'int'>", 2]
View, type, value: ['2.0', "<class 'float'>", 2.0]
View, type, value: ['My string', "<class 'str'>", 'My string']


Fault: <Fault 1: "<class 'Exception'>:Rate limit exceeded. Try again later.">

In [27]:
# Пункт 4: Проверка черного списка
print("Проверка черного списка:")

print(server.black_list_check('ПЕТРОВ', 'ПЕТР', 'ПЕТРОВИЧ', '07.08.1985'))   
print(server.black_list_check('Петрова', 'Пётр', 'Петрович', '07.08.1985')) 
print(server.black_list_check('Светлой','Михаил','Викторович', '13.02.1992'))  
print(server.black_list_check('Петров','Аркадий','Викторович', '01.01.2000')) 

Проверка черного списка:
ПЕТРОВ ПЕТР ПЕТРОВИЧ 07.08.1985 : yes
['Петров Петр Петрович 07.08.1985', 'Петрово Петр Петрович 07.08.1985'] : similar
['Светлый Михаил Викторович 13.02.1992', 'Светлий Михаил Викторович 13.02.1992'] : similar
Петров Аркадий Викторович 01.01.2000 : no


In [28]:
# Подготовка изображений 
img_color = Image.open('Jellyfish.jpg')
img_arr_color = np.array(img_color)
    
img_mono = Image.open('11.bmp')
img_arr_mono = np.array(img_mono)
    
#if img_arr_mono.ndim == 3:
#    img_arr_mono = img_arr_mono[:,:,0]

print("Изображения загружены.")
print(f"Color image shape: {img_arr_color.shape}, dtype: {img_arr_color.dtype}")
print(f"Mono image shape: {img_arr_mono.shape}, dtype: {img_arr_mono.dtype}")

# Сериализация изображений
pimg_color = xmlrpc.client.Binary(pickle.dumps(img_arr_color))
pimg_mono = xmlrpc.client.Binary(pickle.dumps(img_arr_mono))

# Пункт 2: Оценка процента облачности
print("Пункт 2: процент облачности")
threshold = 100
cloud_color = server.cloud_percent(pimg_color, threshold)
cloud_mono  = server.cloud_percent(pimg_mono, threshold)

print(f"Cloud percent (color): {cloud_color:.2f}%")
print(f"Cloud percent (mono):  {cloud_mono:.2f}%")

# Пункт 5: Инверсия
img_bin2_color = server.color_inversion(pimg_color)
img_arr_inv_color = pickle.loads(img_bin2_color.data)
    
img_bin2_mono = server.color_inversion(pimg_mono)
img_arr_inv_mono = pickle.loads(img_bin2_mono.data)
# Пункт 6: Бинаризация
threshold = 128
img_bin2_color_bw = server.binarize_image(pimg_color, threshold)
img_arr_bin_color = pickle.loads(img_bin2_color_bw.data)
    
img_bin2_mono_bw = server.binarize_image(pimg_mono, threshold)
img_arr_bin_mono = pickle.loads(img_bin2_mono_bw.data)
# Пункт 7: Разворот по вертикали 
img_bin2_color_flip = server.flip_vertical(pimg_color)
img_arr_flip_color = pickle.loads(img_bin2_color_flip.data)
    
img_bin2_mono_flip = server.flip_vertical(pimg_mono)
img_arr_flip_mono = pickle.loads(img_bin2_mono_flip.data)
# Отображение всех результатов
figure(figsize=(15, 15))

subplot(4, 2, 1)
imshow(img_arr_color)
title("Original Color")

subplot(4, 2, 2)
imshow(img_arr_mono, cmap='gray')
title("Original Monochrome")

subplot(4, 2, 3)
imshow(img_arr_inv_color)
title("Inverted Color")

subplot(4, 2, 4)
imshow(img_arr_inv_mono, cmap='gray')
title("Inverted Monochrome")

subplot(4, 2, 5)
imshow(img_arr_bin_color)
title(f"Binarized Color Cloud: {cloud_color:.2f}%")    

subplot(4, 2, 6)
imshow(img_arr_bin_mono, cmap='gray')
title(f"Binarized Monochrome Cloud: {cloud_mono:.2f}%") 

subplot(4, 2, 7)
imshow(img_arr_flip_color)
title("Flipped Color")

subplot(4, 2, 8)
imshow(img_arr_flip_mono, cmap='gray')
title("Flipped Monochrome")

show()

print("\nЗаписей в журнале:", len(stats.get_log()))
print("\nВсе события:")
for row in stats.get_log():
    print(" ", row)


Изображения загружены.
Color image shape: (768, 1024, 3), dtype: uint8
Mono image shape: (780, 780), dtype: uint8
Пункт 2: процент облачности
Cloud percent (color): 11.84%
Cloud percent (mono):  7.31%


Fault: <Fault 1: "<class 'Exception'>:Rate limit exceeded. Try again later.">

In [ ]:
print("Только ping")
for row in stats.get_log('ping'):
    print(" ", row)

print("Только cloud_percent")
for row in stats.get_log('cloud_percent'):
    print(" ", row)

print("Только black_list_check")
for row in stats.get_log('black_list_check'):
    print(" ", row)

dt_from = datetime.datetime(2026, 10, 5, 20,00,00,00).strftime("%Y-%m-%d %H:%M:%S")
dt_to   = datetime.datetime(2026, 10, 5, 20,50,00,00).strftime("%Y-%m-%d %H:%M:%S")

print(f"\nЗа последние 30 минут ({dt_from} .. {dt_to})")
for row in stats.get_log(None, dt_from, dt_to):    
    print(" ", row)

print("Комбинированный: за 30 минут")
for row in stats.get_log('flip_vertical', dt_from, dt_to):
    print(" ", row)

In [ ]:
# print(stats.system.listMethods())
# print(stats.ping_stats())

In [ ]:
print("ping →", server.ping())
print("sum  →", server.sum(2, 3))
print("now  →", server.now())
print("Всё работает без stats")

In [29]:
import time

print("=== Rate Limiting ===")
for i in range(1, 9):
    try:
        print(f"{i}:", server.ping())
    except Exception as e:
        print(f"{i}: ОШИБКА — {e}")
    time.sleep(0.1)

print("\n=== too_many_requests из базы ===")
for row in stats2.get_log('too_many_requests'):
    print(row)

=== Rate Limiting ===
1: True
2: True
3: True
4: True
5: True
6: ОШИБКА — <Fault 1: "<class 'Exception'>:Rate limit exceeded. Try again later.">
7: ОШИБКА — <Fault 1: "<class 'Exception'>:Rate limit exceeded. Try again later.">
8: ОШИБКА — <Fault 1: "<class 'Exception'>:Rate limit exceeded. Try again later.">

=== too_many_requests из базы ===
['too_many_requests', '2026-10-19 18:40:44', None]
['too_many_requests', '2026-10-19 18:41:05', None]
['too_many_requests', '2026-10-19 18:41:16', None]
['too_many_requests', '2026-10-19 18:41:16', None]
['too_many_requests', '2026-10-19 18:41:16', None]


In [22]:
print("последние 5 записей:")
for row in stats2.get_log()[::-1][:5]:
    print(row)

последние 5 записей:
['ping', '2026-10-19 18:36:51', 0.028714656829833984]
['ping', '2026-10-19 18:36:51', None]
['ping', '2026-10-19 18:36:51', 0.031778812408447266]
['ping', '2026-10-19 18:36:51', None]
['ping', '2026-10-19 18:36:51', 0.031273603439331055]


In [18]:

print("ТЕСТ Rate Limiting (лимит 5 запросов в 5 сек)")
for i in range(1, 9):
    try:
        print(f"Запрос {i}:", server.ping())
    except Exception as e:
        print(f"Запрос {i}: ОШИБКА — {e}")
    time.sleep(0.1)

ТЕСТ Rate Limiting (лимит 5 запросов в 5 сек)
Запрос 1: True
Запрос 2: True
Запрос 3: True
Запрос 4: True
Запрос 5: True
Запрос 6: ОШИБКА — <Fault 1: "<class 'Exception'>:Rate limit exceeded. Try again later.">
Запрос 7: ОШИБКА — <Fault 1: "<class 'Exception'>:Rate limit exceeded. Try again later.">
Запрос 8: ОШИБКА — <Fault 1: "<class 'Exception'>:Rate limit exceeded. Try again later.">


In [19]:
for row in stats2.get_log('too_many_requests'):
    print(row)

In [ ]:
print("=== 1) Только too_many_requests ===")
for row in stats2.get_log(event_type='too_many_requests'):
    print("  ", row)

print("\n=== 2) Логи за последние 10 минут ===")
now = datetime.datetime.now()
dt_from = (now - datetime.timedelta(minutes=10)).strftime("%Y-%m-%d %H:%M:%S")
dt_to   = now.strftime("%Y-%m-%d %H:%M:%S")
for row in stats2.get_log(dt_from=dt_from, dt_to=dt_to):
    print("  ", row)

print("\n=== 3) Логи 'ping' длительностью > 0.001 сек ===")
for row in stats2.get_log(event_type='ping', min_duration=0.001):
    print("  ", row)

print("\n=== 4) Логи 'cloud_percent' ===")
for row in stats2.get_log(event_type='cloud_percent'):
    print("  ", row)

In [30]:
# 1. too_many_requests (позиционно)
print("=== too_many_requests ===")
for row in stats2.get_log('too_many_requests'):
    print("  ", row)

# 2. За последние 10 минут
print("\n=== за 10 минут ===")
now = datetime.datetime.now()
dt_from = (now - datetime.timedelta(minutes=10)).strftime("%Y-%m-%d %H:%M:%S")
dt_to = now.strftime("%Y-%m-%d %H:%M:%S")
for row in stats2.get_log(None, dt_from, dt_to):
    print("  ", row)

# 3. ping с duration > 0.001
print("\n=== ping с duration > 0.001 ===")
for row in stats2.get_log('ping', None, None, 0.001, None):
    print("  ", row)

# 4. cloud_percent
print("\n=== cloud_percent ===")
for row in stats2.get_log('cloud_percent'):
    print("  ", row)

=== too_many_requests ===
   ['too_many_requests', '2026-10-19 18:40:44', None]
   ['too_many_requests', '2026-10-19 18:41:05', None]
   ['too_many_requests', '2026-10-19 18:41:16', None]
   ['too_many_requests', '2026-10-19 18:41:16', None]
   ['too_many_requests', '2026-10-19 18:41:16', None]

=== за 10 минут ===
   ['ping', '2026-10-19 18:34:19', None]
   ['ping', '2026-10-19 18:34:19', 0.03404808044433594]
   ['now', '2026-10-19 18:34:19', None]
   ['now', '2026-10-19 18:34:19', 0.020082712173461914]
   ['type', '2026-10-19 18:34:19', None]
   ['type', '2026-10-19 18:34:19', 0.019763946533203125]
   ['type', '2026-10-19 18:34:19', None]
   ['type', '2026-10-19 18:34:19', 0.017015457153320312]
   ['type', '2026-10-19 18:34:19', None]
   ['type', '2026-10-19 18:34:19', 0.020814180374145508]
   ['black_list_check', '2026-10-19 18:34:31', None]
   ['black_list_check', '2026-10-19 18:34:31', 0.027361631393432617]
   ['black_list_check', '2026-10-19 18:34:31', None]
   ['black_list_check